In [3]:
# imports 
import math
import numpy as np

# random seed
np.random.seed(67)


## Categorical Cross-Entropy

### Forward pass

$$L = -\sum_k y_k \log(\hat{y}_k)$$

Since $y$ is one-hot, all terms drop except the correct class $c$:

$$L = -\log(\hat{y}_c)$$

### Derivative

$$\frac{\partial L}{\partial \hat{y}_i} = -\frac{y_i}{\hat{y}_i}$$

Since $y_i = 0$ for all wrong classes:

$$\boxed{\frac{\partial L}{\partial \hat{y}_i} = \begin{cases} -\dfrac{1}{\hat{y}_c} & i = c \\ 0 & i \neq c \end{cases}}$$

In vectorised form:

$$\frac{\partial L}{\partial \hat{Y}} = -\frac{Y}{\hat{Y}} \div S$$

Where $S$ is the number of samples (normalisation).


In [4]:
# Common loss class 
class Loss: 
 
    # Calculates the data and regularization losses 
    # given model output and ground truth values 
    def calculate(self, output, y): 
 
        # Calculate sample losses 
        sample_losses = self.forward(output, y) 
 
        # Calculate mean loss 
        data_loss = np.mean(sample_losses) 
 
        # Return loss 
        return data_loss 
 
 
# Cross-entropy loss 
class Loss_CategoricalCrossentropy(Loss): 
 
    # Forward pass 
    def forward(self, y_pred, y_true): 
 
        # Number of samples in a batch 
        samples = len(y_pred) 
 
        # Clip data to prevent division by 0 
        # Clip both sides to not drag mean towards any value 
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7) 
 
        # Probabilities for target values - 
        # only if categorical labels 
        if len(y_true.shape) == 1: 
            correct_confidences = y_pred_clipped[ 
                range(samples), 
                y_true 
            ] 
 
        # Mask values - only for one-hot encoded labels 
        elif len(y_true.shape) == 2: 
            correct_confidences = np.sum( 
                y_pred_clipped * y_true, 
                axis=1 
            ) 
 
        # Losses 
        negative_log_likelihoods = -np.log(correct_confidences) 
        return negative_log_likelihoods 
 
    # Backward pass 
    def backward(self, dvalues, y_true): 
 
        # Number of samples 
        samples = len(dvalues) 
        # Number of labels in every sample 
        # We'll use the first sample to count them 
        labels = len(dvalues[0]) 
 
        # If labels are sparse, turn them into one-hot vector 
        if len(y_true.shape) == 1: 
            y_true = np.eye(labels)[y_true] 
 
        # Calculate gradient 
        self.dinputs = -y_true / dvalues 
        # Normalize gradient 
        self.dinputs = self.dinputs / samples 

In [5]:
y_pred = np.array([
    [0.7, 0.2, 0.1],   # sample 1: 70% class 0, 20% class 1, 10% class 2
    [0.1, 0.8, 0.1],   # sample 2: 10% class 0, 80% class 1, 10% class 2
    [0.2, 0.3, 0.5],   # sample 3: 20% class 0, 30% class 1, 50% class 2
])

y_true = np.array([0, 1, 2])

loss_fn = Loss_CategoricalCrossentropy()
loss = loss_fn.calculate(y_pred, y_true)
print(f"Loss {loss}")

loss_fn.backward(y_pred, y_true)

print(f"Backwards dvalues {loss_fn.dinputs}")


Loss 0.4243218919376292
Backwards dvalues [[-0.47619048 -0.         -0.        ]
 [-0.         -0.41666667 -0.        ]
 [-0.         -0.         -0.66666667]]


> Softmax takes a vector of raw scores (logits) and squashes them into probabilities that sum to 1.

si = exp(xáµ¢) / Î£â±¼ exp(xâ±¼) x = [2.0, 1.0, 0.5] exp(x) = [7.39, 2.72, 1.65] <- numerator for each Î£ exp(x) = 11.76 <- shared denominator s = [0.63, 0.23, 0.14] <- sums to 1.0

## Softmax Activation

### Forward pass

$$s_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

### Why a Jacobian?

Every output $s_i$ depends on every input $x_j$ through the denominator.
The derivative is therefore a matrix $J$ of shape $(n, n)$:

$$J_{ij} = \frac{\partial s_i}{\partial x_j}$$

### Deriving $J$

**Case 1 diagonal** $(i = j)$, quotient rule:

Let $f = e^{x_i}$, $g = \sum_j e^{x_j}$, so $s_i = \dfrac{f}{g}$

$$\frac{\partial s_i}{\partial x_i} = \frac{f'g - fg'}{g^2}
= \frac{e^{x_i} \cdot \sum_j e^{x_j} \;-\; e^{x_i} \cdot e^{x_i}}{\left(\sum_j e^{x_j}\right)^2}$$

$$= \frac{e^{x_i}}{\sum_j e^{x_j}} - \left(\frac{e^{x_i}}{\sum_j e^{x_j}}\right)^2 = s_i - s_i^2 = \boxed{s_i(1 - s_i)}$$

**Case 2 off-diagonal** $(i \neq j)$:

$f = e^{x_i}$ does not depend on $x_j$, so $f' = 0$, but $g' = e^{x_j}$

$$\frac{\partial s_i}{\partial x_j} = \frac{0 \cdot g - e^{x_i} \cdot e^{x_j}}{g^2}
= -\frac{e^{x_i}}{\sum e^{x_j}} \cdot \frac{e^{x_j}}{\sum e^{x_j}} = \boxed{-s_i s_j}$$

### Chain rule

$$\boxed{\frac{\partial L}{\partial x} = J \cdot \delta}$$

Computed per sample since each has its own $s$ and therefore its own $J$.

In [ ]:
class Activation_Softmax: 
    def forward(self, inputs): 
        self.inputs = inputs 
        # Get unnormalized probabilities 
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True)) 
        # Normalize them for each sample 
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True) 
        self.output = probabilities 

    def backward(self, dvalues): 
        # Create uninitialized array 
        self.dinputs = np.empty_like(dvalues) 
 
        # Enumerate outputs and gradients 
        for index, (single_output, single_dvalues) in enumerate(zip(self.output, dvalues)): 
            # Flatten output array 
            single_output = single_output.reshape(-1, 1) 
            # Calculate Jacobian matrix of the output and 
            jacobian_matrix = np.diagflat(single_output) - np.dot(single_output, single_output.T) 
 
            # Calculate sample-wise gradient 
            # and add it to the array of sample gradients 
            self.dinputs[index] = np.dot(jacobian_matrix, 
                                         single_dvalues) 
 

In [7]:
softmax = Activation_Softmax()
softmax.output = y_pred

softmax.backward(loss_fn.dinputs)
print("dinputs (gradient w.r.t logits):", softmax.dinputs)

dinputs (gradient w.r.t logits): [[-0.1         0.06666667  0.03333333]
 [ 0.03333333 -0.06666667  0.03333333]
 [ 0.06666667  0.1        -0.16666667]]


## Combined Softmax + CCE

### The cancellation

Substituting the CCE gradient into the Softmax Jacobian:

$$\frac{\partial L}{\partial x_j} = \sum_k \frac{\partial L}{\partial \hat{y}_k} \cdot J_{kj}
= \sum_k \left(-\frac{y_k}{\hat{y}_k}\right) \cdot \hat{y}_k(\delta_{kj} - \hat{y}_j)$$

The $\hat{y}_k$ cancels:

$$= \sum_k -y_k(\delta_{kj} - \hat{y}_j)
= \underbrace{-\sum_k y_k \delta_{kj}}_{\text{term 1}} + \underbrace{\hat{y}_j \sum_k y_k}_{\text{term 2}}$$

Term 1 â€” Kronecker delta picks out $k = j$ only:

$$-\sum_k y_k \delta_{kj} = -y_j$$

Term 2 â€” $y$ is one-hot so $\sum_k y_k = 1$:

$$\hat{y}_j \sum_k y_k = \hat{y}_j$$

### Result

$$\boxed{\frac{\partial L}{\partial x_j} = \hat{y}_j - y_j}$$

The entire Jacobian disappears. Subtract 1 at the correct class, divide by samples:

$$\frac{\partial L}{\partial X} = \frac{\hat{Y} - Y}{S}$$

### Behaviour at extremes

| $\hat{y}_c$ | Gradient $\hat{y}_c - 1$ | Update |
|---|---|---|
| $0.01$ | $-0.99$ | Large â€” very wrong |
| $0.50$ | $-0.50$ | Medium |
| $0.99$ | $-0.01$ | Small â€” nearly correct |

> **Intuition:** The gradient is self-scaling. A confident correct prediction
> produces almost no update. A confident wrong prediction produces a
> near-maximal correction. No Jacobian loop required.

In [ ]:
# Softmax classifier - combined Softmax activation 
class Activation_Softmax_Loss_CategoricalCrossentropy(): 
    def __init__(self): 
        self.activation = Activation_Softmax() 
        self.loss = Loss_CategoricalCrossentropy() 
 
    def forward(self, inputs, y_true): 
        self.activation.forward(inputs) 
        self.output = self.activation.output 
        return self.loss.calculate(self.output, y_true) 
 
    # Backward pass 
    def backward(self, dvalues, y_true): 
 
        # Number of samples 
        samples = len(dvalues) 
 
        # If labels are one-hot encoded, 
        # turn them into discrete values 
        if len(y_true.shape) == 2: 
            y_true = np.argmax(y_true, axis=1) 
 
        self.dinputs = dvalues.copy() 
        # Calculate gradient 
        self.dinputs[range(samples), y_true] -= 1 
        # Normalize gradient 
        self.dinputs = self.dinputs / samples 

In [9]:
# Combined softmax + loss
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

# Backward: pass y_pred (softmax output from previous cell) + y_true
loss_activation.backward(y_pred, y_true)
print("dinputs:", loss_activation.dinputs)

dinputs: [[-0.1         0.06666667  0.03333333]
 [ 0.03333333 -0.06666667  0.03333333]
 [ 0.06666667  0.1        -0.16666667]]


## ReLU Activation

### Forward pass

$$f(x) = \max(0, x)$$

### Derivative

$$f'(x) = \begin{cases} 1 & x > 0 \\ 0 & x \leq 0 \end{cases}$$

### Chain rule

$$\frac{\partial L}{\partial x_i} = \frac{\partial L}{\partial y_i} \cdot f'(x_i)$$

Since $f'(x_i) \in \{0, 1\}$ this is just a mask:

$$\boxed{\frac{\partial L}{\partial x_i} = \begin{cases} \delta_i & x_i > 0 \\ 0 & x_i \leq 0 \end{cases}}$$

Or in one expression using the indicator function $\mathbf{1}$:

$$\frac{\partial L}{\partial X} = \delta \odot \mathbf{1}[X > 0]$$

Where $\odot$ is element-wise multiplication.

In [10]:
class Activation_ReLU: 
    def forward(self, inputs): 
        self.inputs = inputs # cache inputs
        self.output = np.maximum(0, inputs) 

    def backward(self, dvalues): 
        # since we need to modify original variable, 
        # let's make a copy of values first 
        self.dinputs = dvalues.copy() 
 
        # zero gradient where input values were negative 
        # this is just the derivative of max(0, inputs)
        self.dinputs[self.inputs <= 0] = 0 

In [11]:
# Simulate hidden layer output (pre-ReLU), shape (3 samples, 5 neurons)
hidden_inputs = np.array([[ 1.2, -0.5,  0.8, -1.1,  0.3],
                          [-0.2,  0.9,  1.5,  0.4, -0.7],
                          [ 0.6,  1.1, -0.3,  0.2,  0.9]])

relu = Activation_ReLU()
relu.forward(hidden_inputs)      # caches inputs, zeros out negatives

# dvalues coming back from the layer above (e.g. dense2.dinputs)
# for now use a placeholder gradient of same shape
dvalues_from_above = np.ones((3, 5)) * 0.1

relu.backward(dvalues_from_above)
print("ReLU dinputs:", relu.dinputs)

ReLU dinputs: [[0.1 0.  0.1 0.  0.1]
 [0.  0.1 0.1 0.1 0. ]
 [0.1 0.1 0.  0.1 0.1]]


## Dense Layer

### Forward pass

$$\text{output} = \text{inputs} \cdot W + b$$

### Deriving `dweights`

Single element: $\text{out}_{s,j} = \sum_i X_{s,i} \cdot W_{i,j} + b_j$

$$\frac{\partial \text{out}_{s,j}}{\partial W_{i,j}} = X_{s,i}
\quad \Rightarrow \quad
\frac{\partial L}{\partial W_{i,j}} = \sum_s \frac{\partial L}{\partial \text{out}_{s,j}} \cdot X_{s,i}$$

$$\boxed{\frac{\partial L}{\partial W} = X^T \cdot \delta}$$

where delta is the dvalues

### Deriving `dbiases`

$$\frac{\partial \text{out}_{s,j}}{\partial b_j} = 1
\quad \Rightarrow \quad
\boxed{\frac{\partial L}{\partial b} = \sum_s \delta_{s,j}}$$

### Deriving `dinputs`

$$\frac{\partial \text{out}_{s,j}}{\partial X_{s,i}} = W_{i,j}
\quad \Rightarrow \quad
\frac{\partial L}{\partial X_{s,i}} = \sum_j \delta_{s,j} \cdot W_{i,j}$$

$$\boxed{\frac{\partial L}{\partial X} = \delta \cdot W^T}$$

### Summary

$$\frac{\partial L}{\partial W} = X^T \cdot \delta, \qquad
\frac{\partial L}{\partial b} = \sum_s \delta, \qquad
\frac{\partial L}{\partial X} = \delta \cdot W^T$$


In [ ]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.10 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

    def forward(self, inputs):
        self.inputs = inputs # cache input
        self.output = np.dot(inputs, self.weights) + self.biases

    # dvalues comes from the end point of network / from loss_fn output
    def backward(self, dvalues):
        # gradient weights: how much each weight contributed to the error
        self.dweights = np.dot(self.inputs.T, dvalues)
        # gradient biases: just sum the incoming gradients
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)
        # gradient inputs: what we pass back to the previous layer
        self.dinputs = np.dot(dvalues, self.weights.T)


In [ ]:
X = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
])

dense = Layer_Dense(3, 5)
dense.forward(X) # caches X, produces output shape (3, 5)

# Backward: relu.dinputs is the gradient flowing back from ReLU
dense.backward(relu.dinputs)

print("dweights:\n", dense.dweights)   # shape (3, 5): one per weight
print("dbiases:\n",  dense.dbiases)    # shape (1, 5): one per neuron
print("dinputs:\n",  dense.dinputs)    # shape (3, 3): passed to prev layer


dweights:
 [[0.8 1.1 0.5 1.1 0.8]
 [1.  1.3 0.7 1.3 1. ]
 [1.2 1.5 0.9 1.5 1.2]]
dbiases:
 [[0.2 0.2 0.2 0.2 0.2]]
dinputs:
 [[-0.00141395 -0.00488857 -0.01467297]
 [ 0.0025998   0.00740872 -0.04413779]
 [ 0.02349103 -0.02189164 -0.02610405]]


## Full Picture

In [ ]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.10 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

    def forward(self, inputs):
        self.inputs = inputs # cache input
        self.output = np.dot(inputs, self.weights) + self.biases

    # dvalues comes from the end point of network / from loss_fn output
    def backward(self, dvalues):
        # gradient weights: how much each weight contributed to the error
        self.dweights = np.dot(self.inputs.T, dvalues)
        # gradient biases: just sum the incoming gradients
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)
        # gradient inputs: what we pass back to the previous layer
        self.dinputs = np.dot(dvalues, self.weights.T)

class Activation_ReLU: 
    def forward(self, inputs): 
        self.inputs = inputs # cache inputs
        self.output = np.maximum(0, inputs) 

    def backward(self, dvalues): 
        # since we need to modify original variable, 
        # let's make a copy of values first 
        self.dinputs = dvalues.copy() 
 
        # zero gradient where input values were negative 
        # this is just the derivative of max(0, inputs)
        self.dinputs[self.inputs <= 0] = 0 

# Common loss class 
class Loss: 
    def calculate(self, output, y): 
        sample_losses = self.forward(output, y) 
        data_loss = np.mean(sample_losses) 
        return data_loss 
 
# Cross-entropy loss 
class Loss_CategoricalCrossentropy(Loss): 
 
    # Forward pass 
    def forward(self, y_pred, y_true): 
 
        # Number of samples in a batch 
        samples = len(y_pred) 
 
        # Clip data to prevent division by 0 
        # Clip both sides to not drag mean towards any value 
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7) 
 
        # Probabilities for target values - 
        # only if categorical labels 
        if len(y_true.shape) == 1: 
            correct_confidences = y_pred_clipped[ 
                range(samples), 
                y_true 
            ] 
 
        # Mask values - only for one-hot encoded labels 
        elif len(y_true.shape) == 2: 
            correct_confidences = np.sum( 
                y_pred_clipped * y_true, 
                axis=1 
            ) 
 
        # Losses 
        negative_log_likelihoods = -np.log(correct_confidences) 
        return negative_log_likelihoods 
 
    # Backward pass 
    def backward(self, dvalues, y_true): 
 
        # Number of samples 
        samples = len(dvalues) 
        # Number of labels in every sample 
        # We'll use the first sample to count them 
        labels = len(dvalues[0]) 
 
        # If labels are sparse, turn them into one-hot vector 
        if len(y_true.shape) == 1: 
            y_true = np.eye(labels)[y_true] 
 
        # Calculate gradient 
        self.dinputs = -y_true / dvalues 
        # Normalize gradient 
        self.dinputs = self.dinputs / samples 

class Activation_Softmax: 
    def forward(self, inputs): 
        self.inputs = inputs 
        # Get unnormalized probabilities 
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True)) 
        # Normalize them for each sample 
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True) 
        self.output = probabilities 

    def backward(self, dvalues): 
        # Create uninitialized array 
        self.dinputs = np.empty_like(dvalues) 
 
        # Enumerate outputs and gradients 
        for index, (single_output, single_dvalues) in enumerate(zip(self.output, dvalues)): 
            # Flatten output array 
            single_output = single_output.reshape(-1, 1) 
            # Calculate Jacobian matrix of the output and 
            jacobian_matrix = np.diagflat(single_output) - np.dot(single_output, single_output.T) 
 
            # Calculate sample-wise gradient 
            # and add it to the array of sample gradients 
            self.dinputs[index] = np.dot(jacobian_matrix, 
                                         single_dvalues) 
 
# Softmax classifier - combined Softmax activation 
class Activation_Softmax_Loss_CategoricalCrossentropy(): 
    def __init__(self): 
        self.activation = Activation_Softmax() 
        self.loss = Loss_CategoricalCrossentropy() 
 
    # Forward pass 
    def forward(self, inputs, y_true): 
        # Output layer's activation function 
        self.activation.forward(inputs) 
        # Set the output 
        self.output = self.activation.output 
        # Calculate and return loss value 
        return self.loss.calculate(self.output, y_true) 
 
    # Backward pass 
    def backward(self, dvalues, y_true): 
 
        # Number of samples 
        samples = len(dvalues) 
 
        # If labels are one-hot encoded, 
        # turn them into discrete values 
        if len(y_true.shape) == 2: 
            y_true = np.argmax(y_true, axis=1) 
 
        self.dinputs = dvalues.copy() 
        # Calculate gradient 
        self.dinputs[range(samples), y_true] -= 1 
        # Normalize gradient 
        self.dinputs = self.dinputs / samples 
